# 欠損値・外れ値の検出と方針

欠損値と外れ値の検出方法、代表的な対処方針、そしてデータリークを避ける適用順序を学びます。

> **Kaggle Tutorial** 対応ノートブック（`/learn/python-data-analysis/07-missing-and-outliers`）です。上から順にセルを実行（`Shift + Enter` または「ランタイム → すべてのセルを実行」）して動作を確認できます。

In [ ]:
# [Colab / ローカル環境用セットアップ]
# Kaggle Notebook 上では既存の /kaggle/input/titanic をそのまま使用し、
# Google Colab やローカル環境では互換スキーマの Titanic サンプルデータ（891行 / 418行）を自動配置します。
# ※ Kaggle 本番データで提出する場合は kaggle CLI (!kaggle competitions download -c titanic) をご利用ください。
import os
import numpy as np
import pandas as pd

def _ensure_titanic_data():
    target_dir = "/kaggle/input/titanic"
    if os.path.exists(os.path.join(target_dir, "train.csv")):
        return

    try:
        os.makedirs(target_dir, exist_ok=True)
        data_dir = target_dir
    except OSError:
        data_dir = os.path.abspath("./.kaggle_input/titanic")
        os.makedirs(data_dir, exist_ok=True)
        _orig_read_csv = pd.read_csv

        def _compat_read_csv(filepath_or_buffer, *args, **kwargs):
            if isinstance(filepath_or_buffer, str) and filepath_or_buffer.startswith("/kaggle/input/"):
                rel = filepath_or_buffer[len("/kaggle/input/"):]
                local_path = os.path.abspath(os.path.join("./.kaggle_input", rel))
                if os.path.exists(local_path):
                    filepath_or_buffer = local_path
            return _orig_read_csv(filepath_or_buffer, *args, **kwargs)

        pd.read_csv = _compat_read_csv

    rng = np.random.default_rng(42)
    surnames = ["Braund", "Cumings", "Heikkinen", "Futrelle", "Allen", "Moran", "McCarthy", "Palsson", "Johnson", "Nasser"]
    given_m = ["Owen", "William", "James", "Henry", "Leonard", "Timothy", "charles", "George"]
    given_f = ["Florence", "Laina", "Lily", "elisabeth", "Adele", " Marguerite", "Anna", "Mary"]

    def _make_split(n, start_id, with_target):
        pclass = rng.choice([1, 2, 3], size=n, p=[0.24, 0.21, 0.55])
        sex = rng.choice(["male", "female"], size=n, p=[0.65, 0.35])
        age = np.clip(rng.normal(29.7, 14.5, size=n), 0.42, 80.0).round(1)
        sibsp = rng.choice([0, 1, 2, 3, 4], size=n, p=[0.68, 0.23, 0.05, 0.02, 0.02])
        parch = rng.choice([0, 1, 2, 3], size=n, p=[0.76, 0.13, 0.09, 0.02])
        base_fare = np.where(pclass == 1, 60.0, np.where(pclass == 2, 20.0, 11.0))
        fare = (base_fare * rng.lognormal(0.0, 0.55, size=n)).round(2)
        embarked = rng.choice(["S", "C", "Q"], size=n, p=[0.72, 0.19, 0.09]).astype(object)

        names = []
        cabins = []
        tickets = []
        for i in range(n):
            sn = surnames[i % len(surnames)]
            if sex[i] == "male":
                title = "Master" if age[i] < 13 else "Mr"
                gn = given_m[i % len(given_m)].strip().capitalize()
            else:
                title = "Miss" if age[i] < 24 else "Mrs"
                gn = given_f[i % len(given_f)].strip().capitalize()
            names.append(f"{sn}, {title}. {gn}")
            cabins.append(f"{'ABC'[pclass[i] - 1]}{10 + (i % 90)}" if rng.random() > 0.77 else np.nan)
            tickets.append(f"PC {17500 + (i % 350)}" if pclass[i] == 1 else str(230000 + i))

        age_obj = age.astype(float)
        age_obj[rng.random(n) < 0.20] = np.nan
        embarked[rng.random(n) < 0.005] = np.nan
        if not with_target:
            fare[0] = np.nan

        data = {
            "PassengerId": np.arange(start_id, start_id + n),
            "Pclass": pclass,
            "Name": names,
            "Sex": sex,
            "Age": age_obj,
            "SibSp": sibsp,
            "Parch": parch,
            "Ticket": tickets,
            "Fare": fare,
            "Cabin": pd.Series(cabins, dtype=object),
            "Embarked": pd.Series(embarked, dtype=object),
        }
        if with_target:
            family_size = sibsp + parch + 1
            logit = (
                -1.65
                + np.where(sex == "female", 2.45, 0.0)
                + np.where(pclass == 1, 1.35, np.where(pclass == 2, 0.35, -0.55))
                + np.where(np.nan_to_num(age_obj, nan=29.0) < 12, 1.2, 0.0)
                + np.where((family_size >= 2) & (family_size <= 4), 0.75, -0.35)
            )
            prob = 1.0 / (1.0 + np.exp(-logit))
            survived = (rng.random(n) < prob).astype(int)
            ordered = {"PassengerId": data["PassengerId"], "Survived": survived}
            for k, v in data.items():
                if k != "PassengerId":
                    ordered[k] = v
            return pd.DataFrame(ordered)
        return pd.DataFrame(data)

    _make_split(891, 1, True).to_csv(os.path.join(data_dir, "train.csv"), index=False)
    _make_split(418, 892, False).to_csv(os.path.join(data_dir, "test.csv"), index=False)
    print(f"Titanic dataset ready at: {data_dir}")

_ensure_titanic_data()

## 欠損値を検出する

In [ ]:
import pandas as pd

df = pd.read_csv("/kaggle/input/titanic/train.csv")

df.isnull().mean().sort_values(ascending=False)  # 列ごとの欠損率

欠損は「なぜ欠けているか」で意味が変わります。
ランダムな欠損もあれば、**欠けていること自体が情報**（例：未記入＝該当なし）の場合もあります。

## 欠損値への方針

| 方針                 | 使いどころ                                         | 注意点                           |
| -------------------- | -------------------------------------------------- | -------------------------------- |
| **行・列の削除**     | 欠損が少数、または欠損率が極端に高い列             | 情報を失う・本番データに使えない |
| **代入（補完）**     | 多くのケース。数値は中央値、カテゴリは最頻値が無難 | 分布を歪めうる                   |
| **欠損フラグの追加** | 欠損そのものが意味を持つとき                       | 列が増える                       |

In [ ]:
# 中央値で補完しつつ、欠損だったことをフラグとして残す
df["Age_missing"] = df["Age"].isnull().astype(int)
df["Age"] = df["Age"].fillna(df["Age"].median())

## 外れ値を検出する

代表的な手法は **IQR 法**と **z-score** です。

- **IQR 法**：第 1 四分位 $Q_1$、第 3 四分位 $Q_3$、$\text{IQR} = Q_3 - Q_1$ とし、
  $Q_1 - 1.5 \times \text{IQR}$ を下回る／$Q_3 + 1.5 \times \text{IQR}$ を上回る値を外れ値とみなす
- **z-score**：平均からの標準偏差単位の距離。目安として絶対値が 3 を超えると外れ値

$$
z = \frac{x - \mu}{\sigma}
$$

In [ ]:
q1, q3 = df["Fare"].quantile([0.25, 0.75])
iqr = q3 - q1
lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
outliers = df[(df["Fare"] < lower) | (df["Fare"] > upper)]

In [ ]:
print(f"IQR bounds: [{lower:.2f}, {upper:.2f}], detected outliers: {len(outliers)} rows")

## 外れ値への方針

- **除去**：明らかな入力ミス（ありえない値）のみ
- **クリップ（上下限で丸める）**：極端値の影響を抑えつつ件数を保つ
- **対数変換など**：右に裾を引く分布（運賃・価格など）を圧縮する

> **Warning:** 外れ値は安易に消さないこと。本物のレアケースを捨てると、本番で似た例を取りこぼします。 「ノイズか、意味のある稀少値か」を EDA で見極めてから方針を決めます。

## リークを避ける適用順序

> **Warning:** 補完の中央値や外れ値のしきい値は、**訓練データだけ**から計算し、検証・テストへ適用します。 全データから統計量を計算すると、検証側の情報が漏れるデータリークになり、 スコアが過大評価されます。これらの前処理は交差検証の各 fold の**内側**で行うのが原則です。

## このトラックのまとめ

- NumPy のベクトル化と broadcasting で、数値処理を高速・簡潔に書ける
- pandas で読み込み・選択・集計・結合・変換まで一通り扱える
- 可視化と EDA で「効く特徴」「欠損・外れ値」「リークの兆候」を見つけられる
- 前処理は訓練データから学び、fold の内側で適用してデータリークを防ぐ

## 次のステップ

次トラック（L2）では、いよいよ**機械学習の基礎**（教師あり学習・モデルの学習と評価）に進みます。
ここで身につけたデータ操作が、その土台になります。